In [1]:
import requests

In [2]:
import pandas as pd

# green = pd.read_parquet('../nyc_taxi_data/green_tripdata_2025-01.parquet')
# yellow = pd.read_parquet('../nyc_taxi_data/yellow_tripdata_2025-01.parquet')
# fhvhv = pd.read_parquet('../nyc_taxi_data/fhvhv_tripdata_2025-01.parquet')
# fhv = pd.read_parquet('../nyc_taxi_data/fhv_tripdata_2025-01.parquet')

# for name, dataset in [("green", green), ("yellow", yellow), ("fhvhv", fhvhv), ("fhv", fhv)]:
#     print(f"{name}: shape={dataset.shape}")
#     print(f"{name} columns: {list(dataset.columns)}\n")


In [ ]:
import pyarrow.parquet as pq
import pyarrow
import pyarrowfs_adlgen2
from dotenv import load_dotenv
import os

load_dotenv()


ADLS_NAME = os.getenv("ADLS_NAME")
ADLS_KEY =  os.getenv("ADLS_KEY")
directory = "sources"  # ou ajuste conforme necessário

import json
import os
from tqdm import tqdm
json_path = "parquet_row_counts.json"

# Carrega o dicionário existente ou cria um novo
if os.path.exists(json_path):
    with open(json_path, "r") as f:
        row_counts = json.load(f)
else:
    row_counts = {}


from azure.storage.blob import BlobServiceClient

# Create the container client
account_url = f"https://{ADLS_NAME}.blob.core.windows.net"
credential = ADLS_KEY
container_name = "taxi"  # Change if your container name is different

handler = pyarrowfs_adlgen2.AccountHandler.from_account_name(ADLS_NAME, ADLS_KEY)
fs = pyarrow.fs.PyFileSystem(handler)

blob_service_client = BlobServiceClient(account_url=account_url, credential=credential)
container_client = blob_service_client.get_container_client(container_name)

blobs = container_client.list_blobs(name_starts_with='source/')
for blob in tqdm(blobs):
    if blob.name.endswith('.parquet') and blob.name not in row_counts:
        pf = pq.ParquetFile(f"taxi/{blob.name}", filesystem=fs)
        row_counts[blob.name] = pf.metadata.num_rows


# Salva o dicionário atualizado
with open(json_path, "w") as f:
    json.dump(row_counts, f, indent=2)

total_rows = sum(row_counts.values())
print(f"Total de linhas em todos os arquivos parquet: {total_rows}")


0it [00:00, ?it/s]

578it [00:08, 67.02it/s] 

Total de linhas em todos os arquivos parquet: 3945621967


In [4]:
from collections import defaultdict
import json
import re

def extract_mm_yyyy(fname):
    # Exemplo: source/yellow/2025/yellow_tripdata_2025-03.parquet
    match = re.search(r'(\d{4})-(\d{2})', fname)
    if match:
        return f"{match.group(2)}-{match.group(1)}"
    return fname

yellow_schemas = {}
yellow_schema_diffs = []

# Percorra todos os arquivos do yellow_trip
for blob in container_client.list_blobs(name_starts_with='source/yellow/'):
    if blob.name.endswith('.parquet'):
        pf = pq.ParquetFile(f"taxi/{blob.name}", filesystem=fs)
        schema = pf.schema.to_arrow_schema()
        yellow_schemas[blob.name] = schema

# Ordena os arquivos por nome (que contém ano/mês)
sorted_files = sorted(yellow_schemas.keys())

prev_schema = None
prev_fname = None
for fname in sorted_files:
    schema = yellow_schemas[fname]
    if prev_schema is not None and schema != prev_schema:
        added = [f for f in schema.names if f not in prev_schema.names]
        removed = [f for f in prev_schema.names if f not in schema.names]
        type_changes = []
        for col in set(schema.names).intersection(prev_schema.names):
            prev_type = str(prev_schema.field(col).type)
            curr_type = str(schema.field(col).type)
            if prev_type != curr_type:
                type_changes.append({
                    "column": col,
                    "from": prev_type,
                    "to": curr_type
                })
        yellow_schema_diffs.append({
            "from": extract_mm_yyyy(prev_fname),
            "to": extract_mm_yyyy(fname),
            "added_columns": added,
            "removed_columns": removed,
            "type_changes": type_changes
        })
    prev_schema = schema
    prev_fname = fname

# Salva as diferenças em um arquivo JSON simples
with open("yellow_schema_diffs_simple.json", "w", encoding="utf-8") as f:
    json.dump(yellow_schema_diffs, f, indent=2, ensure_ascii=False)

# Exemplo de visualização
for diff in yellow_schema_diffs:
    print(diff)

{'from': '12-2009', 'to': '01-2010', 'added_columns': ['vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'pickup_longitude', 'pickup_latitude', 'rate_code', 'store_and_fwd_flag', 'dropoff_longitude', 'dropoff_latitude', 'payment_type', 'fare_amount', 'tip_amount', 'tolls_amount', 'total_amount'], 'removed_columns': ['vendor_name', 'Trip_Pickup_DateTime', 'Trip_Dropoff_DateTime', 'Passenger_Count', 'Trip_Distance', 'Start_Lon', 'Start_Lat', 'Rate_Code', 'store_and_forward', 'End_Lon', 'End_Lat', 'Payment_Type', 'Fare_Amt', 'Tip_Amt', 'Tolls_Amt', 'Total_Amt'], 'type_changes': []}
{'from': '01-2010', 'to': '02-2010', 'added_columns': ['__index_level_0__'], 'removed_columns': [], 'type_changes': [{'column': 'rate_code', 'from': 'string', 'to': 'int64'}]}
{'from': '03-2010', 'to': '04-2010', 'added_columns': [], 'removed_columns': ['__index_level_0__'], 'type_changes': [{'column': 'rate_code', 'from': 'int64', 'to': 'string'}]}
{'from': '12-2010', 'to'

In [6]:
import json
from collections import Counter

with open("parquet_row_counts.json", "r") as f:
    count = f.read()


    data = json.loads(count)


    yellow_counts_by_year = Counter()
    for k in data:
        if k.startswith("source/yellow/2014"):
            yellow_counts_by_year["2014"] += 1
        elif k.startswith("source/yellow/2015"):
            yellow_counts_by_year["2015"] += 1
        elif k.startswith("source/yellow/2016"):
            yellow_counts_by_year["2016"] += 1
        elif k.startswith("source/yellow/2017"):
            yellow_counts_by_year["2017"] += 1
        elif k.startswith("source/yellow/2018"):
            yellow_counts_by_year["2018"] += 1
        elif k.startswith("source/yellow/2019"):
            yellow_counts_by_year["2019"] += 1
        elif k.startswith("source/yellow/2020"):
            yellow_counts_by_year["2020"] += 1
        elif k.startswith("source/yellow/2021"):
            yellow_counts_by_year["2021"] += 1
        elif k.startswith("source/yellow/2022"):
            yellow_counts_by_year["2022"] += 1
        elif k.startswith("source/yellow/2023"):
            yellow_counts_by_year["2023"] += 1
        elif k.startswith("source/yellow/2024"):
            yellow_counts_by_year["2024"] += 1
        elif k.startswith("source/yellow/2025"):
            yellow_counts_by_year["2025"] += 1

    total_yellow = sum(yellow_counts_by_year.values())
    print(f"Total source/yellow de 2014 até 2025: {total_yellow}")
    print("Quantidade por ano:")
    for year in sorted(yellow_counts_by_year):
        print(f"{year}: {yellow_counts_by_year[year]}")

Total source/yellow de 2014 até 2025: 131
Quantidade por ano:
2014: 8
2015: 12
2016: 12
2017: 12
2018: 12
2019: 12
2020: 12
2021: 12
2022: 12
2023: 12
2024: 12
2025: 3


In [9]:
def get_yellow_taxi_data(year_month, limit=None):
    """
    Busca e retorna o dataset do yellow taxi para um determinado ano-mês do ADLS.
    
    Args:
        year_month (str): Formato 'YYYY-MM' (ex: '2013-05')
        limit (int, optional): Número máximo de linhas a retornar. Se None, retorna todas.
    
    Returns:
        pandas.DataFrame: Dataset do yellow taxi
    """
    # Constrói o caminho do arquivo
    file_path = f"source/yellow/{year_month[:4]}/yellow_tripdata_{year_month}.parquet"
    
    # Lê o arquivo parquet do ADLS
    pf = pq.ParquetFile(f"taxi/{file_path}", filesystem=fs)
    
    if limit:
        # Lê apenas as primeiras 'limit' linhas
        df = pf.read(use_pandas_metadata=True).slice(0, limit).to_pandas()
    else:
        # Lê o arquivo completo
        df = pf.read().to_pandas()
    
    return df

# Exemplo de uso: buscar dados de maio de 2013 (primeiras 1000 linhas)
yellow_2013_05 = get_yellow_taxi_data('2018-05')
print(f"Shape do dataset: {yellow_2013_05.shape}")
print(f"Colunas: {list(yellow_2013_05.columns)}")
print(f"\nPrimeiras 5 linhas:")
yellow_2013_05.head()

Shape do dataset: (9224788, 19)
Colunas: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']

Primeiras 5 linhas:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2018-05-01 00:13:56,2018-05-01 00:22:46,1,1.6,1,N,230,50,1,8.0,0.5,0.5,1.85,0.0,0.3,11.15,None,None
1,1,2018-05-01 00:23:26,2018-05-01 00:29:56,1,1.7,1,N,263,239,1,7.5,0.5,0.5,2.00,0.0,0.3,10.80,None,None
2,1,2018-05-01 00:36:23,2018-05-01 00:48:26,2,2.6,1,N,239,152,1,12.0,0.5,0.5,1.00,0.0,0.3,14.30,None,None
3,1,2018-05-01 00:26:12,2018-05-01 00:27:05,1,0.0,1,N,145,145,1,2.5,0.5,0.5,9.63,0.0,0.3,13.43,None,None
4,1,2018-05-01 00:29:51,2018-05-01 00:30:02,1,0.0,1,N,145,145,2,2.5,0.5,0.5,0.00,0.0,0.3,3.80,None,None


In [12]:
len(yellow_2013_05)

9224788

In [57]:
total_mem = 0

# Recrie o iterador de blobs
blobs = container_client.list_blobs(name_starts_with='source/')

for blob in blobs:
    # print(blob.name)
    total_mem += blob.size

print(f"Total de memória em todos os arquivos parquet: {total_mem/(1024)**3:2f} GBs")


Total de memória em todos os arquivos parquet: 34.506272 GBs


In [ ]:
import pandas as pd

df = pd.read_parquet('nyc_taxi_data/yellow_tripdata_2025-01.parquet').head(5000)


In [3]:
# Convert df to match the desired schema
df_converted = df.copy()

# Add the new columns
df_converted['pickup_year_month'] = df_converted['tpep_pickup_datetime'].dt.strftime('%Y-%m')
df_converted['ingestion_timestamp'] = pd.Timestamp.now()

# Rename airport_fee column to match schema (lowercase)
df_converted = df_converted.rename(columns={'Airport_fee': 'airport_fee'})

# Convert data types to match the schema as closely as possible in pandas
df_converted['VendorID'] = df_converted['VendorID'].astype('int8')  # ByteType equivalent
df_converted['passenger_count'] = df_converted['passenger_count'].astype('int32')  # IntegerType
df_converted['RatecodeID'] = df_converted['RatecodeID'].astype('int8')  # ByteType equivalent
df_converted['PULocationID'] = df_converted['PULocationID'].astype('int16')  # ShortType equivalent
df_converted['DOLocationID'] = df_converted['DOLocationID'].astype('int16')  # ShortType equivalent
df_converted['payment_type'] = df_converted['payment_type'].astype('int8')  # ByteType equivalent

# Convert decimal fields to appropriate float types (pandas doesn't have exact decimal equivalents)
df_converted['fare_amount'] = df_converted['fare_amount'].round(2).astype('float64')
df_converted['extra'] = df_converted['extra'].round(2).astype('float64')
df_converted['mta_tax'] = df_converted['mta_tax'].round(2).astype('float64')
df_converted['tip_amount'] = df_converted['tip_amount'].round(2).astype('float64')
df_converted['tolls_amount'] = df_converted['tolls_amount'].round(2).astype('float64')
df_converted['improvement_surcharge'] = df_converted['improvement_surcharge'].round(2).astype('float64')
df_converted['total_amount'] = df_converted['total_amount'].round(2).astype('float64')
df_converted['congestion_surcharge'] = df_converted['congestion_surcharge'].round(2).astype('float64')
df_converted['airport_fee'] = df_converted['airport_fee'].round(2).astype('float64')

# Save to parquet with the converted schema
df_converted.to_parquet('sample.parquet', index=False)

In [5]:
from pyarrow import parquet as pq
def get_parquet_schema(file_path):
    """
    Obtém o schema de um arquivo parquet local.
    
    Args:
        file_path (str): Caminho para o arquivo parquet local
    
    Returns:
        pyarrow.Schema: Schema do arquivo parquet
    """
    pf = pq.ParquetFile(file_path)
    return pf.schema.to_arrow_schema()

# Exemplo de uso com o arquivo que foi salvo anteriormente
schema = get_parquet_schema('nyc_taxi_data/test_schema.parquet')

print("Schema do arquivo parquet:")
print(schema)
print("\nColunas e tipos:")
for i, field in enumerate(schema):
    print(f"{i}: {field.name} - {field.type}")

Schema do arquivo parquet:
VendorID: int32
tpep_pickup_datetime: timestamp[us]
tpep_dropoff_datetime: timestamp[us]
passenger_count: int64
trip_distance: double
RatecodeID: int64
store_and_fwd_flag: large_string
PULocationID: int32
DOLocationID: int32
payment_type: int64
fare_amount: double
extra: double
mta_tax: double
tip_amount: double
tolls_amount: double
improvement_surcharge: double
total_amount: double
congestion_surcharge: double
Airport_fee: double
cbd_congestion_fee: double

Colunas e tipos:
0: VendorID - int32
1: tpep_pickup_datetime - timestamp[us]
2: tpep_dropoff_datetime - timestamp[us]
3: passenger_count - int64
4: trip_distance - double
5: RatecodeID - int64
6: store_and_fwd_flag - large_string
7: PULocationID - int32
8: DOLocationID - int32
9: payment_type - int64
10: fare_amount - double
11: extra - double
12: mta_tax - double
13: tip_amount - double
14: tolls_amount - double
15: improvement_surcharge - double
16: total_amount - double
17: congestion_surcharge - doubl

In [3]:

import pandas as pd
df = pd.read_parquet('nyc_taxi_data/yellow_tripdata_05.parquet')
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2020-05-01 00:02:28,2020-05-01 00:18:07,1.0,0.0,1.0,N,234,256,1,12.2,3.0,0.5,2.4,0.0,0.3,18.4,2.5,None
1,1,2020-05-01 00:23:21,2020-05-01 00:26:01,2.0,0.4,1.0,N,264,264,1,4.0,0.5,0.5,0.5,0.0,0.3,5.8,0.0,None
2,1,2020-05-01 00:54:58,2020-05-01 00:57:11,1.0,0.3,1.0,N,264,264,2,3.5,0.5,0.5,0.0,0.0,0.3,4.8,0.0,None
3,1,2020-05-01 00:07:10,2020-05-01 00:12:46,1.0,1.7,1.0,N,107,229,2,7.0,3.0,0.5,0.0,0.0,0.3,10.8,2.5,None
4,1,2020-05-01 00:55:47,2020-05-01 01:01:54,0.0,0.9,1.0,N,237,262,1,6.0,3.0,0.5,1.2,0.0,0.3,11.0,2.5,None


In [9]:
green = pd.read_parquet('nyc_taxi_data/green_tripdata_2025-03.parquet')

# Convert green dataset to match the desired schema
green_converted = green.copy()

# Add the pickup_year_month column
# green_converted['pickup_year_month'] = green_converted['lpep_pickup_datetime'].dt.strftime('%Y-%m')

# Convert data types to match the schema as closely as possible in pandas
green_converted['VendorID'] = green_converted['VendorID'].astype('int8')  # TINYINT equivalent
green_converted['passenger_count'] = green_converted['passenger_count'].astype('Int8')  # TINYINT equivalent (nullable)
green_converted['RatecodeID'] = green_converted['RatecodeID'].astype('Int8')  # TINYINT equivalent (nullable)
green_converted['PULocationID'] = green_converted['PULocationID'].astype('int16')  # SMALLINT equivalent
green_converted['DOLocationID'] = green_converted['DOLocationID'].astype('int16')  # SMALLINT equivalent
green_converted['payment_type'] = green_converted['payment_type'].astype('Int8')  # TINYINT equivalent (nullable)
green_converted['trip_type'] = green_converted['trip_type'].astype('Int8')  # TINYINT equivalent (nullable)

# Convert decimal fields to appropriate float types (pandas doesn't have exact decimal equivalents)
green_converted['fare_amount'] = green_converted['fare_amount'].round(2).astype('float64')
green_converted['extra'] = green_converted['extra'].round(2).astype('float64')
green_converted['mta_tax'] = green_converted['mta_tax'].round(2).astype('float64')
green_converted['tip_amount'] = green_converted['tip_amount'].round(2).astype('float64')
green_converted['tolls_amount'] = green_converted['tolls_amount'].round(2).astype('float64')
green_converted['improvement_surcharge'] = green_converted['improvement_surcharge'].round(2).astype('float64')
green_converted['total_amount'] = green_converted['total_amount'].round(2).astype('float64')
green_converted['congestion_surcharge'] = green_converted['congestion_surcharge'].round(2).astype('float64')
green_converted['ehail_fee'] = green_converted['ehail_fee'].round(2).astype('float64')

# Handle cbd_congestion_fee (not in target schema, so we'll drop it)
green_converted = green_converted.drop(columns=['cbd_congestion_fee'])

# Reorder columns to match the schema order
column_order = [
    'VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime', 'passenger_count',
    'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID',
    'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
    'improvement_surcharge', 'total_amount', 'trip_type', 'congestion_surcharge',
    'ehail_fee'
]

green_converted = green_converted[column_order]



In [10]:
green_converted.head().to_parquet('green_tripdata_sample.parquet', index=False)